In [88]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [89]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [90]:
descriptivo = pd.read_csv('/content/drive/MyDrive/Datasets/Riesgo_ergonomico/descriptivo2.csv', sep=',')

In [ ]:
descriptivo.head()

,Age,Gender,Occupation,Years_in_occupation,Working_hours_per_day,Working_days_per_week,Years_using_driving_apps,App_used,Working_hours_per_day_numeric
0,1,1,3,2,2,1,1,6,2
1,2,2,1,2,2,2,2,1,2
2,2,2,1,2,2,2,2,6,2
3,2,1,3,1,2,2,3,3,2
4,2,1,1,3,3,2,3,6,3


In [91]:
descriptivo.shape


(231, 9)

In [92]:
# Remove the column named 'NaN' if it exists
if 'NaN' in descriptivo.columns:
    descriptivo = descriptivo.drop(columns=['NaN'])

# Display the DataFrame head to confirm the column is removed
display(descriptivo.head())

,Age,Gender,Occupation,Years_in_occupation,Working_hours_per_day,Working_days_per_week,Years_using_driving_apps,App_used,Working_hours_per_day_numeric
0,1,1,3,2,2,1,1,6,2
1,2,2,1,2,2,2,2,1,2
2,2,2,1,2,2,2,2,6,2
3,2,1,3,1,2,2,3,3,2
4,2,1,1,3,3,2,3,6,3


In [93]:
display(descriptivo.describe())

,Age,Gender,Occupation,Years_in_occupation,Working_hours_per_day,Working_days_per_week,Years_using_driving_apps,App_used,Working_hours_per_day_numeric
count,231.000000,231.000000,231.000000,231.000000,231.000000,231.000000,231.000000,231.000000,231.000000
mean,1.696970,1.796537,1.290043,2.203463,2.424242,1.956710,1.891775,3.792208,2.424242
std,0.530741,0.403448,0.602773,0.873367,0.633912,0.533924,0.612530,2.214602,0.633912
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,2.000000,1.000000,2.000000,2.000000,2.000000,2.000000,1.500000,2.000000
50%,2.000000,2.000000,1.000000,2.000000,3.000000,2.000000,2.000000,3.000000,3.000000
75%,2.000000,2.000000,1.000000,3.000000,3.000000,2.000000,2.000000,6.000000,3.000000
max,3.000000,2.000000,3.000000,4.000000,3.000000,3.000000,3.000000,6.000000,3.000000


In [94]:
# Save the descriptive statistics to a CSV file
descriptive_stats.to_csv('descriptivo_estadisticas.csv', index=True)
print("Descriptive statistics saved to 'descriptivo_estadisticas.csv'")

Descriptive statistics saved to 'descriptivo_estadisticas.csv'


In [95]:
import openpyxl
output_path = '/content/drive/MyDrive/Datasets/Riesgo_ergonomico/descriptivo_estadisticas.xlsx'
descriptive_stats.to_excel(output_path, index=True)
print(f"Descriptive statistics saved to '{output_path}'")

Descriptive statistics saved to '/content/drive/MyDrive/Datasets/Riesgo_ergonomico/descriptivo_estadisticas.xlsx'


### Análisis Estadístico Descriptivo (todas las columnas)

In [96]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns

# Create Output widget to display plots
output_two_cols = widgets.Output()

def plot_two_columns(col1, col2):
    with output_two_cols:
        clear_output(wait=True)
        plt.figure(figsize=(10, 6))

        # Heuristic to treat int64 columns with <= 20 unique values as categorical
        is_categorical_col1 = descriptivo[col1].nunique() <= 20 and descriptivo[col1].dtype == 'int64'
        is_categorical_col2 = descriptivo[col2].nunique() <= 20 and descriptivo[col2].dtype == 'int64'

        if col1 == col2: # Single column distribution
            if is_categorical_col1:
                sns.countplot(x=col1, data=descriptivo, palette='viridis', hue=col1, legend=False)
                plt.title(f'Distribución de {col1}')
                plt.ylabel('Conteo')
            else:
                sns.histplot(descriptivo[col1], kde=True, bins='auto')
                plt.title(f'Distribución de {col1}')
                plt.ylabel('Frecuencia')
        elif is_categorical_col1 and is_categorical_col2: # Two categorical columns
            # Create a cross-tabulation for stacked bar chart
            cross_tab = pd.crosstab(descriptivo[col1], descriptivo[col2])
            cross_tab.plot(kind='bar', stacked=True, figsize=(10, 6))
            plt.title(f'Conteo de {col2} por {col1}')
            plt.ylabel('Conteo')
            plt.xticks(rotation=45, ha='right')
        elif (is_categorical_col1 and not is_categorical_col2): # col1 categorical, col2 numerical
            sns.boxplot(x=col1, y=col2, data=descriptivo, palette='viridis', hue=col1, legend=False)
            plt.title(f'{col2} por {col1}')
            plt.ylabel(col2)
            plt.xticks(rotation=45, ha='right')
        elif (not is_categorical_col1 and is_categorical_col2): # col1 numerical, col2 categorical
            sns.boxplot(x=col2, y=col1, data=descriptivo, palette='viridis', hue=col2, legend=False)
            plt.title(f'{col1} por {col2}')
            plt.ylabel(col1)
            plt.xticks(rotation=45, ha='right')
        else: # Two numerical columns
            sns.scatterplot(x=col1, y=col2, data=descriptivo)
            plt.title(f'Relación entre {col1} y {col2}')
            plt.ylabel(col2)

        plt.xlabel(col1)
        plt.tight_layout()
        plt.show()

# Create dropdown widgets
column_selector_1 = widgets.Dropdown(
    options=descriptivo.columns.tolist(),
    description='Columna 1:',
    disabled=False,
)

column_selector_2 = widgets.Dropdown(
    options=descriptivo.columns.tolist(),
    description='Columna 2:',
    disabled=False,
)

# Display the widgets and output area
display(widgets.HBox([column_selector_1, column_selector_2]), output_two_cols)

# Link the dropdowns to the plotting function
widgets.interactive_output(
    plot_two_columns,
    {'col1': column_selector_1, 'col2': column_selector_2}
);

Output()